-------------------------------
-------------------------------
# Laboratorio #6 - IA (CC3085)
* Dulce Ambrosio - 231143
* Daniel Chet - 231177
* Gadiel Ocaña - 231270

-------------------------------
-------------------------------

-------------------------------
## Task 2.1 - Diseño del juego (Connect Four)
-------------------------------

In [13]:
# Inicialización y Constantes
# Importaciones necesarias
import numpy as np
import math
import random
import time
from IPython.display import clear_output

# Constantes del juego
ROWS = 6
COLS = 7

PLAYER = 1
AI = 2
EMPTY = 0

In [14]:
# Creación del tablero y manejo de movimientos
def create_board():
    return np.zeros((ROWS, COLS), dtype=int)

# Actions (posbles acciones) - Diapositiva 6
# Una columna es valida si la posicion superior esta vacia
def get_valid_moves(board):
    valid_moves = []
    for col in range(COLS):
        if board[ROWS-1][col] == 0:
            valid_moves.append(col)
    return valid_moves

# Succ(s,a), se aplican los movimiento - Diapositiva 6
def drop_piece(board, col, piece):
    new_board = board.copy()
    for row in range(ROWS):
        if new_board[row][col] == 0:
            new_board[row][col] = piece
            break
    return new_board

In [15]:
# Detección de victoria (IsEnd)
# IsEnd(s), se verifica la victoria - Diapositiva 6
def winning_move(board, piece):
    # Horizontal
    for c in range(COLS-3):
        for r in range(ROWS):
            if board[r][c] == piece and board[r][c+1] == piece and board[r][c+2] == piece and board[r][c+3] == piece:
                return True
    # Vertical
    for c in range(COLS):
        for r in range(ROWS-3):
            if board[r][c] == piece and board[r+1][c] == piece and board[r+2][c] == piece and board[r+3][c] == piece:
                return True
    # Diagonal positiva (/)
    for c in range(COLS-3):
        for r in range(ROWS-3):
            if board[r][c] == piece and board[r+1][c+1] == piece and board[r+2][c+2] == piece and board[r+3][c+3] == piece:
                return True
    # Diagonal negativa (\)
    for c in range(COLS-3):
        for r in range(3, ROWS):
            if board[r][c] == piece and board[r-1][c+1] == piece and board[r-2][c+2] == piece and board[r-3][c+3] == piece:
                return True
    return False

# Estado terminal - Diapositiva 8
# Verificación de condición de victoria.
# Esto permite identificar estados terminales del juego.
def is_terminal(board):
    return winning_move(board, PLAYER) or winning_move(board, AI) or len(get_valid_moves(board)) == 0

-------------------------------
## Task 2.2 - Se implementa la Poda Alfa-Beta
-------------------------------

In [16]:
# Placeholder - Esta función será detallada en el Task 2.3
# Función evaluate(board) - Esta función recorre todo el tablero buscando ventanas de 4. - Diapositiva 37
# Implementación de Eval(s).
# Esta heurística estima qué tan favorable es el tablero
# cuando no se puede llegar a un estado terminal.
def evaluate(board, piece):
    return 0 

# Minimax con poda alpha-beta - Diapositiva 43
# Versión optimizada de Minimax usando poda Alpha-Beta.
# Los valores alpha y beta representan los límites inferior
# y superior de los nodos MAX y MIN respectivamente.
# Si alpha >= beta se puede podar la rama.
def alphabeta(board, depth, alpha, beta, maximizing):
    is_term = is_terminal(board)
    if depth == 0 or is_term:
        if is_term:
            if winning_move(board, AI): return (None, 10000000)
            elif winning_move(board, PLAYER): return (None, -10000000)
            else: return (None, 0)
        else:
            return (None, evaluate(board, AI)) # Llama al placeholder

    valid_moves = get_valid_moves(board)
    if maximizing:
        value = -math.inf
        best_col = random.choice(valid_moves)
        for col in valid_moves:
            temp_board = drop_piece(board, col, AI)
            new_score = alphabeta(temp_board, depth-1, alpha, beta, False)[1]
            if new_score > value:
                value = new_score
                best_col = col
            alpha = max(alpha, value)
            if alpha >= beta: break
        return best_col, value
    else:
        value = math.inf
        best_col = random.choice(valid_moves)
        for col in valid_moves:
            temp_board = drop_piece(board, col, PLAYER)
            new_score = alphabeta(temp_board, depth-1, alpha, beta, True)[1]
            if new_score < value:
                value = new_score
                best_col = col
            beta = min(beta, value)
            if alpha >= beta: break
        return best_col, value

-------------------------------
## Task 2.3 - Función evaluate(board)
-------------------------------

In [17]:
# Evaluar una ventana de 4 posiciones - Diapositiva 37
# Evaluación heurística de una ventana de 4 posiciones.
# Forma parte de la función Eval(s) usada cuando la
# búsqueda se corta en profundidad limitada.

def evaluate_window(window, piece):
    score = 0
    opp = PLAYER if piece == AI else AI
    if window.count(piece) == 4: score += 100
    elif window.count(piece) == 3 and window.count(EMPTY) == 1: score += 5
    elif window.count(piece) == 2 and window.count(EMPTY) == 2: score += 2
    if window.count(opp) == 3 and window.count(EMPTY) == 1: score -= 4
    return score

# Función evaluate(board) - Esta función recorre todo el tablero buscando ventanas de 4. - Diapositiva 37
# Implementación de Eval(s).
# Esta heurística estima qué tan favorable es el tablero
# cuando no se puede llegar a un estado terminal.
def evaluate(board, piece):
    score = 0
    # Prioridad centro
    center_array = [int(i) for i in list(board[:, COLS//2])]
    score += center_array.count(piece) * 3

    # Puntaje Horizontal, Vertical y Diagonales
    for r in range(ROWS):
        row_array = [int(i) for i in list(board[r,:])]
        for c in range(COLS-3):
            score += evaluate_window(row_array[c:c+4], piece)
    for c in range(COLS):
        col_array = [int(i) for i in list(board[:,c])]
        for r in range(ROWS-3):
            score += evaluate_window(col_array[r:r+4], piece)
    for r in range(ROWS-3):
        for c in range(COLS-3):
            window = [board[r+i][c+i] for i in range(4)]
            score += evaluate_window(window, piece)
    for r in range(3, ROWS):
        for c in range(COLS-3):
            window = [board[r-i][c+i] for i in range(4)]
            score += evaluate_window(window, piece)
    return score

In [18]:
def print_board_visual(board):
    clear_output(wait=True)
    board_view = np.flip(board, 0)
    print("\n\t=== CONECTA 4 (Lab 6) ===")
    print("\tHumano: 🔴  |  IA: 🟡\n")
    for row in board_view:
        print("\t|" + "|".join([" 🔴 " if c==1 else " 🟡 " if c==2 else " ⚫ " for c in row]) + "|")
    print("\t-----------------------------")
    print("\t  0   1   2   3   4   5   6\n")

def play_game():
    board = create_board()
    turn = random.randint(PLAYER, AI)
    print_board_visual(board)
    
    while not is_terminal(board):
        if turn == PLAYER:
            col = int(input("Tu turno (0-6): "))
            if col in get_valid_moves(board):
                board = drop_piece(board, col, PLAYER)
                turn = AI
        else:
            print("IA pensando...")
            time.sleep(0.5)
            col, _ = alphabeta(board, 5, -math.inf, math.inf, True)
            board = drop_piece(board, col, AI)
            turn = PLAYER
        print_board_visual(board)
    
    if winning_move(board, PLAYER): print("¡Ganaste!")
    elif winning_move(board, AI): print("La IA ganó.")
    else: print("Empate.")

play_game()


	=== CONECTA 4 (Lab 6) ===
	Humano: 🔴  |  IA: 🟡

	| ⚫ | ⚫ | ⚫ | ⚫ | ⚫ | ⚫ | ⚫ |
	| ⚫ | ⚫ | ⚫ | 🟡 | ⚫ | ⚫ | ⚫ |
	| 🟡 | 🟡 | ⚫ | 🔴 | ⚫ | ⚫ | ⚫ |
	| 🔴 | 🟡 | 🟡 | 🟡 | 🔴 | 🟡 | ⚫ |
	| 🔴 | 🔴 | 🔴 | 🟡 | 🔴 | 🔴 | ⚫ |
	| 🔴 | 🔴 | 🟡 | 🟡 | 🟡 | 🔴 | ⚫ |
	-----------------------------
	  0   1   2   3   4   5   6

La IA ganó.
